<a href="https://colab.research.google.com/github/manluz555-ops/Line_progr/blob/main/HW_9_Manzar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашня робота № 9

Манзар Л.В.

In [5]:
import gymnasium as gym
import numpy as np

env = gym.make("FrozenLake-v1", is_slippery=True)
n_states = env.observation_space.n
n_actions = env.action_space.n

# доступ до перехідних ймовірностей
P = env.unwrapped.P

def compute_value_function(policy, gamma=0.99, theta=1e-8):
    V = np.zeros(n_states)
    while True:
        delta = 0
        for s in range(n_states):
            a = policy[s]
            v = 0
            for prob, next_state, reward, done in P[s][a]:
                v += prob * (reward + gamma * V[next_state])
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        if delta < theta:
            break
    return V

def policy_iteration(gamma=0.99):
    policy = np.random.choice(n_actions, size=n_states)
    stable = False
    while not stable:
        V = compute_value_function(policy, gamma)
        stable = True
        for s in range(n_states):
            old_action = policy[s]
            action_values = np.zeros(n_actions)
            for a in range(n_actions):
                for prob, next_state, reward, done in P[s][a]:
                    action_values[a] += prob * (reward + gamma * V[next_state])
            policy[s] = np.argmax(action_values)
            if old_action != policy[s]:
                stable = False
    return policy, V

optimal_policy, optimal_V = policy_iteration()

def show_render(policy):
    actions = ["←", "↓", "→", "↑"]
    grid_size = int(np.sqrt(n_states))
    for i in range(grid_size):
        row = policy[i*grid_size:(i+1)*grid_size]
        print(" ".join(actions[a] for a in row))
print("Оптимальна політика:")
show_render(optimal_policy)

# --- тут вставляємо симуляцію ---
def run_episode(env, policy, max_steps=100):
    state, _ = env.reset()
    total_reward = 0
    for step in range(max_steps):
        action = policy[state]
        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break
    return total_reward

# Запускаємо кілька епізодів і дивимось середній результат
episodes = 20
results = [run_episode(env, optimal_policy) for _ in range(episodes)]
print("Середня винагорода за 20 епізодів:", np.mean(results))


print("Оптимальна політика:")
show_render(optimal_policy)



Оптимальна політика:
← ↑ ↑ ↑
← ← ← ←
↑ ↓ ← ←
← → ↓ ←
Середня винагорода за 20 епізодів: 0.8
Оптимальна політика:
← ↑ ↑ ↑
← ← ← ←
↑ ↓ ← ←
← → ↓ ←


## Висновки

1. Алгоритм ітерації політик успішно знайшов оптимальну стратегію для середовища **FrozenLake-v1**.  
2. Візуалізація політики у вигляді стрілок показала логічний шлях агента до цільового стану, з униканням небезпечних клітинок.  
3. Перевірка на практиці (20 епізодів) дала середню винагороду **0.7**, що підтверджує ефективність знайденої політики.  
4. Отримані результати демонструють, що ітерація політик є дієвим методом для дискретних середовищ підкріплювального навчання.  

### Загальний підсумок
Модель не лише теоретично знайшла оптимальну політику, але й практично показала високий рівень успішності у досягненні цілі. Це підтверджує правильність реалізації алгоритму та його застосування для задач підкріплювального навчання.
